# Python API

The BEM interface follows the usual NGSolve form language and mirrors the mathematical notation for boundary integral operators.

## Imports

```python
from ngsolve import *
from ngsolve.bem import *
```

## Boundary Spaces

Common scalar trace spaces:

```python
fes_l2 = SurfaceL2(mesh, order=order-1, dual_mapping=True)
fes_h1 = H1(mesh, order=order)

u, v = fes_l2.TnT()
u_h1, v_h1 = fes_h1.TnT()
```

- `SurfaceL2`: boundary density space, used for Neumann-type traces,
- `H1`: volume space whose boundary trace is used for Dirichlet-type traces,
- vector-valued operators use spaces such as `VectorH1`, `HDivSurface`, and `HCurl` traces,
- triangular and quadrilateral surface elements are supported.

## Kernels and Layer Potentials

Laplace kernel:

$$
  G_0(x,y) = \frac{1}{4\pi |x-y|}.
$$

Helmholtz outgoing kernel:

$$
  G_\kappa(x,y) = \frac{e^{i\kappa |x-y|}}{4\pi |x-y|}.
$$

Single-layer potential:

$$
  (V\rho)(x) = \int_\Gamma G(x,y)\,\rho(y)\,d\sigma_y.
$$

Double-layer potential:

$$
  (K\mu)(x) = \int_\Gamma
  \frac{\partial G(x,y)}{\partial n_y}\,\mu(y)\,d\sigma_y.
$$

This is the naming convention behind `LaplaceSL`, `LaplaceDL`, `HelmholtzSL`, and `HelmholtzDL`.

## From Paper to Code

A single-layer bilinear form

$$
  \langle V\rho_h, \eta_h \rangle_\Gamma
$$

is written as

```python
V = LaplaceSL(u * ds) * v * ds
```

`u * ds` marks the source density on the boundary, `LaplaceSL(...)` applies the layer potential, and multiplication by `v * ds` tests the result on the boundary.

## Laplace Operators

Single-layer operator:

```python
V = LaplaceSL(u * ds) * v * ds
```

Double-layer operator:

```python
K = LaplaceDL(u_h1 * ds) * v * ds
```

Boundary mass matrix:

```python
M = BilinearForm(u_h1 * v * ds).Assemble()
```

Dirichlet-to-Neumann direct formulation:

$$
  V j = \left(\frac12 M + K\right)m.
$$

NGSolve form:

```python
rhs = ((0.5 * M.mat + K.mat) * dirichlet.vec).Evaluate()
neumann.vec.data = solvers.CGSolver(V.mat, pre) * rhs
```

## Helmholtz Operators

```python
kappa = 4.0

V = HelmholtzSL(u * ds, kappa) * v * ds
K = HelmholtzDL(u * ds, kappa) * v * ds
C = HelmholtzCF(u * ds, kappa) * v * ds
```

- `HelmholtzSL`: single-layer operator,
- `HelmholtzDL`: double-layer operator,
- `HelmholtzCF`: combined-field operator for formulations such as Brakhage-Werner.

## Other Implemented Operators

```python
L = LameSL(u * ds, E=2.0, nu=0.25) * v * ds
```

```python
n = specialcf.normal(3)
D = MaxwellDL(Cross(u, n) * ds, kappa) * v.Trace() * ds
```

Available building blocks:

- Laplace: `LaplaceSL`, `LaplaceDL`,
- Helmholtz: `HelmholtzSL`, `HelmholtzDL`, `HelmholtzCF`,
- Lamé elasticity: `LameSL`,
- Maxwell: `MaxwellDL`; single-layer-type terms use vector-valued Helmholtz potentials.

Vector-valued potentials and product spaces work as usual in NGSolve.

## Potential Evaluation

A potential operator can be evaluated from a grid function density:

```python
SL = LaplaceSL(u * ds)
potential = SL(gfu)
```

Evaluation on a target boundary region can use a local expansion:

```python
potential_on_screen = SL(gfu, target_boundary)
```

Differentiated potentials use the operator interface:

```python
grad_potential = grad(SL)(gfu)
```